In [138]:
! pip install langchain-google-genai google-generativeai langchain-core requests pydantic

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
#Tool Create 
@tool
def multiply(a:int , b:int)->int:
    """Given 2 number a and b this tool returns their product"""
    return a*b
    

In [4]:
print(multiply.invoke({'a':3,'b':4}))

12


In [5]:
multiply.name
multiply.description

'Given 2 number a and b this tool returns their product'

In [6]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
#tool binding
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [10]:
llm_with_tools=llm.bind_tools([multiply])

In [12]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing great! I'm ready to help you with any questions you have. How can I assist you today?\n", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-3b96c4b6-30c7-49ea-b933-c76c6262f392-0')

In [77]:
user_query=HumanMessage("can you multiply 6 and 7 for me?")

In [78]:
message=[user_query]

In [79]:
message

[HumanMessage(content='can you multiply 6 and 7 for me?')]

In [80]:
# results=llm_with_tools.invoke('can you multiply 6 and 7 for me?').tool_calls[0]

In [81]:
result=llm_with_tools.invoke('can you multiply 6 and 7 for me?')

In [82]:
result

AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 6.0, "b": 7.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-84a0777f-0036-48bc-bdc4-0ede91d5cbf2-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6.0, 'b': 7.0}, 'id': '1d409be0-ef8f-47a1-ab1b-a010735bbb6e', 'type': 'tool_call'}])

In [83]:
message.append(result)

In [84]:
message

[HumanMessage(content='can you multiply 6 and 7 for me?'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 6.0, "b": 7.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-84a0777f-0036-48bc-bdc4-0ede91d5cbf2-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6.0, 'b': 7.0}, 'id': '1d409be0-ef8f-47a1-ab1b-a010735bbb6e', 'type': 'tool_call'}])]

In [85]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 6.0, 'b': 7.0},
 'id': '1d409be0-ef8f-47a1-ab1b-a010735bbb6e',
 'type': 'tool_call'}

In [86]:
# multiply.invoke({'a':3,'b':4})

tool_result = multiply.invoke(result.tool_calls[0])

In [87]:
message.append(tool_result)

In [88]:
message

[HumanMessage(content='can you multiply 6 and 7 for me?'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 6.0, "b": 7.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-84a0777f-0036-48bc-bdc4-0ede91d5cbf2-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6.0, 'b': 7.0}, 'id': '1d409be0-ef8f-47a1-ab1b-a010735bbb6e', 'type': 'tool_call'}]),
 ToolMessage(content='42', name='multiply', tool_call_id='1d409be0-ef8f-47a1-ab1b-a010735bbb6e')]

In [89]:
llm_with_tools.invoke(message).content

'The product of 6 and 7 is 42.\n'

In [ ]:

#New tools Creation

In [90]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [91]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1766880001,
 'time_last_update_utc': 'Sun, 28 Dec 2025 00:00:01 +0000',
 'time_next_update_unix': 1766966401,
 'time_next_update_utc': 'Mon, 29 Dec 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 89.947}

In [93]:
convert.invoke({'base_currency_value':10, 'conversion_rate':89.947})

899.47

In [112]:
#tool binding
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [113]:
llm_with_tools_currency=llm.bind_tools([get_conversion_factor, convert])

In [ ]:
llm_with_tools_currency

RunnableBinding(bound=ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001D6A216D810>, async_client=<google.ai.generativelanguage_v1beta.services.generative_service.async_client.GenerativeServiceAsyncClient object at 0x000001D6A2F2F110>, default_metadata=()), kwargs={'tools': [{'function_declarations': [{'name': 'get_conversion_factor', 'description': 'This function fetches the currency conversion factor between a given base currency and a target currency', 'parameters': {'type': 'object', 'properties': {'target_currency': {'type': 'string'}, 'base_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency']}}, {'name': 'convert', 'description': 'given a currency conversion rate this function calculates the target currency value from a given base currency value', 'parameters': {'type': 'object', 'properties': {'base_currency_value': {'type': '

In [ ]:
messages=[HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [115]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [135]:
ai_message=llm_with_tools_currency.invoke(messages)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 10.167867923s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}

KeyboardInterrupt: 

In [117]:
messages.append(ai_message)

In [127]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-30104ed6-5aac-4365-9802-ba476cb056b7-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'USD', 'base_currency': 'INR'}, 'id': 'c468df74-b67f-44b4-b090-5d12e0d2b575', 'type': 'tool_call'}]),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-30104ed6-5aac-4365-9802-ba476cb056b7-0', tool_calls=[{'name'

In [128]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': 'c468df74-b67f-44b4-b090-5d12e0d2b575',
  'type': 'tool_call'}]

In [130]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [131]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-30104ed6-5aac-4365-9802-ba476cb056b7-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'USD', 'base_currency': 'INR'}, 'id': 'c468df74-b67f-44b4-b090-5d12e0d2b575', 'type': 'tool_call'}]),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-30104ed6-5aac-4365-9802-ba476cb056b7-0', tool_calls=[{'name'

In [134]:
llm_with_tools.invoke(messages).content

'The conversion factor from INR to USD is 0.01112. I cannot convert 10 INR to USD with the available tools.'

In [ ]:

import pydantic
from langchain.agents import create_structured_chat_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_structured_chat_agent(
    llm=llm,
    tools=[get_conversion_factor, convert],
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[get_conversion_factor, convert],
    verbose=True
)

In [141]:
import pydantic
# print pydantic version in a robust way
version = getattr(pydantic, "__version__", None) or getattr(pydantic, "VERSION", None)
if version is None:
	try:
		from importlib.metadata import version as _pkg_version
		version = _pkg_version("pydantic")
	except Exception:
		version = "unknown"
print(version)

2.12.5


In [ ]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})